# Tragedy: "Epicness" by Character

In [1]:
import numpy as np
import pandas as pd

homer_df = pd.read_parquet("../parquet/homer_dtm.parquet")
tragedy_df = pd.read_parquet("../parquet/tragedy-with-years_dtm.parquet")
log_ratio_df = pd.read_parquet("../parquet/epicness_log_ratio.parquet")

In [4]:
epicness = log_ratio_df.set_index("lemma")["epicness"]
speaker_avg = tragedy_df.multiply(epicness, axis=0).sum() / tragedy_df.sum()
speaker_df = speaker_avg.reset_index()
speaker_df.columns = ["dramatist", "title", "speaker", "epicness"]

In [5]:
speaker_df

,dramatist,title,speaker,epicness
0,Aeschylus,Agamemnon,Αἴγισθος,-1.074031
1,Aeschylus,Agamemnon,Κασάνδρα,-1.330435
2,Aeschylus,Agamemnon,Κλυταιμήστρα,-1.088413
3,Aeschylus,Agamemnon,Κῆρυξ,-1.246929
4,Aeschylus,Agamemnon,Φύλαξ,-1.016871
...,...,...,...,...
297,Sophocles,Trachiniae,Ἡμιχόριον,-0.870091
298,Sophocles,Trachiniae,Ἡμιχόριον 1,-1.815401
299,Sophocles,Trachiniae,Ἡμιχόριον 2,-2.000839
300,Sophocles,Trachiniae,Ἡρακλῆς,-1.109848


In [7]:
import altair as alt

speaker_df["label"] = speaker_df["dramatist"] + " – " + speaker_df["title"] + " – " + speaker_df["speaker"]

alt.Chart(speaker_df).mark_point().encode(
    x=alt.X("epicness:Q", title="Average epicness"),
    y=alt.Y("label:N", sort="-x", title=None, axis=alt.Axis(labelLimit=300)),
    color="dramatist:N"
).properties(width=600, height=3000)

alt.Chart(...)

In [8]:
import re

def classify(speaker):
    s = speaker.strip()
    if re.match(r"(Χορ|Ἡμιχ|Ἡμίχο|Προπομπ|Ημιχ)", s):
        return "chorus"
    if re.match(r"(Ἄγγελος|Ἐξάγγελος|Κῆρυξ|Λίχας|Ταλθύβιος|Αγγελος)", s):
        return "messenger"
    return "other"

speaker_df["speaker_class"] = speaker_df["speaker"].apply(classify)

In [12]:
epicness = log_ratio_df.set_index("lemma")["epicness"]

def classify(speaker):
    s = speaker.strip()
    if re.match(r"(Χορ|Ἡμιχ|Ἡμίχο|Προπομπ|Ημιχ)", s):
        return "chorus"
    if re.match(r"(Ἄγγελος|Ἐξάγγελος|Κῆρυξ|Λίχας|Ταλθύβιος|Αγγελος)", s):
        return "messenger"
    return "other"

class_counts = tragedy_df.T
class_counts["class"] = class_counts.index.get_level_values("speaker").map(classify)
class_counts = class_counts.groupby("class").sum().T

class_avg = class_counts.multiply(epicness, axis=0).sum() / class_counts.sum()
class_avg = class_avg.reset_index(name="epicness")

In [13]:
alt.Chart(class_avg).mark_point().encode(
    x=alt.X("epicness:Q"),
    y=alt.Y("class:N", sort="-x", title=None),
    color="class:N",
).properties(width=600, height=200)

alt.Chart(...)